# Study 804 — Realized-Kurtosis Premium 🎲📊

**Do stocks with fat-tailed recent returns earn a cross-sectional premium?**

Amaya, Christoffersen, Jacobs & Vasquez (2015) — the paper famous for the negative
realized-**skewness** relation — also tests realized **kurtosis** (a name's recent
fat-tailedness, the fourth moment) and finds it a **weak / ambiguous** predictor, mostly
subsumed by skewness and volatility. We take the self-contained daily version on a liquid
US cross-section (2010-01-04 → 2026-06-30, 50 names) and sort **long the
high-kurt / short the low-kurt** names.

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — magnitudes are an upper
bound.*


## 1. The idea in one picture

**Kurtosis** is how *fat-tailed* a return distribution is — how often a stock throws a big move (either direction) versus a calm one. A high-kurtosis name lives in fits and starts; a low-kurtosis name grinds. If investors dislike (or love) fat tails, kurtosis might be priced. But kurtosis is *symmetric* — it mixes up-tail and down-tail — so whatever premium a tail carries is mostly already captured by **skewness** (which side the tail is on) and **volatility** (how big it is). That is why the paper flags it as the *weak* one.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=1.78, t_nw=1.79, hi_bps=8.76, lo_bps=6.97, gross_sharpe=0.43)
print('long high-kurt / short low-kurt spread: %+.2f bps/day (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  high-kurt book %+.2f bps vs low-kurt book %+.2f bps'
      % (R['hi_bps'], R['lo_bps']))
print('  gross spread Sharpe (before cost): %.2f' % R['gross_sharpe'])

long high-kurt / short low-kurt spread: +1.78 bps/day (NW t = +1.79)
  high-kurt book +8.76 bps vs low-kurt book +6.97 bps
  gross spread Sharpe (before cost): 0.43


## 2. Is the sort just lucky? A live synthetic control

We plant the effect in a seeded toy world (`edge>0`, fat-tailed names earn more) and check the detector recovers it — and that it stays *silent* on the null (`edge=0`, kurtosis present but unpriced). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from realized_kurtosis import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=804, n_assets=40, n_days=1500))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.010, seed=804, n_assets=40, n_days=1500))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up)' % planted['t_nw'])

null world   : spread NW t = +0.81  (should be ~0)
planted world: spread NW t = +12.81  (should light up)


## 3. The honest verdict — the edge is *weak*, exactly as the paper says

On this liquid mega-cap tape the long-high-kurt / short-low-kurt spread is **+1.78 bps/day** with NW *t* = **+1.79** — the *right sign* (high-kurt names did edge out low-kurt ones), but **below the |t| ≥ 2 bar** the desk requires to call an edge real. The pooled book Welch *t* is a limp **+0.69**, and the whole (weak) effect lives in the second half of the sample — it is a flat **zero** in 2010–2017 (*t* = +0.11). The seeded synthetic control recovers a *planted* relation cleanly and never fires on the null, so this is a faithful weak measurement, not a bug. **Signal: Weak** (right-signed but sub-threshold), **Tradability: Mirage** — the +1.78 bps gross edge is smaller than the 2.14 bps/day round-trip cost at even 1 bp one-way.